# Colab T4 Open Source LLM Endpoint

This notebook runs a small open-source instruct model on Colab T4 and exposes it over a temporary HTTP URL.

The local Streamlit app calls it with:

```env
COLAB_LLM_URL=https://your-temporary-url.trycloudflare.com
COLAB_LLM_TIMEOUT=90
```


## 1. Check GPU

`Runtime > Change runtime type > T4 GPU`

In [ ]:
!nvidia-smi

## 2. The Text Model Stack




In [ ]:
# Keep Colab's Torch/CUDA stack as-is. Only pin the Hugging Face text stack.
!pip -q uninstall -y transformers tokenizers accelerate torchvision huggingface-hub safetensors
!pip -q install --no-cache-dir --no-deps \
  "transformers==4.48.3" \
  "tokenizers==0.21.0" \
  "accelerate==1.2.1" \
  "huggingface-hub==0.27.1" \
  "safetensors==0.5.2"

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("Done. Restart runtime now")


## 3. Versions After Restart


In [ ]:
import transformers
import tokenizers
import accelerate
import huggingface_hub
import safetensors

print("transformers", transformers.__version__, transformers.__file__)
print("tokenizers", tokenizers.__version__)
print("accelerate", accelerate.__version__)
print("huggingface_hub", huggingface_hub.__version__)
print("safetensors", safetensors.__version__)


## 4. Load The Model

Default: `Qwen/Qwen2.5-1.5B-Instruct`.

For stronger answer style - `Qwen/Qwen2.5-3B-Instruct`.


In [ ]:
import os

os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TRANSFORMERS_NO_VISION"] = "1"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Loaded", MODEL_ID)


## 5. The Local HTTP Server

The server has two endpoints:

- `GET /health`
- `POST /generate`



In [ ]:
import json
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread

import torch

HOST = "0.0.0.0"
PORT = 8000
SYSTEM_PROMPT = (
    "You are a careful mutual fund factsheet assistant. "
    "Answer only from the provided context. "
    "Do not provide investment advice. "
    "If the answer is not present, say the context is insufficient."
)


def build_answer(prompt, max_new_tokens=350):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


class ChatHandler(BaseHTTPRequestHandler):
    def log_message(self, *_args):
        return

    def send_json(self, status, payload):
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path.startswith("/health"):
            self.send_json(200, {"status": "ok", "model": MODEL_ID})
        else:
            self.send_json(404, {"error": "not found"})

    def do_POST(self):
        if not self.path.startswith("/generate"):
            self.send_json(404, {"error": "not found"})
            return

        try:
            length = int(self.headers.get("Content-Length", "0"))
            payload = json.loads(self.rfile.read(length).decode("utf-8"))
            answer = build_answer(
                payload.get("prompt", ""),
                int(payload.get("max_new_tokens", 350)),
            )
            self.send_json(200, {"answer": answer})
        except Exception as exc:
            self.send_json(500, {"error": str(exc)})


server = ThreadingHTTPServer((HOST, PORT), ChatHandler)
Thread(target=server.serve_forever, daemon=True).start()
print(f"Server running on http://localhost:{PORT}")


## 6. The Local Endpoint In Colab


In [ ]:
import requests

test_payload = {
    "prompt": "Answer from context only. Question: What is NAV? Context: Page 1: NAV means Net Asset Value.",
    "max_new_tokens": 120,
}

response = requests.post("http://localhost:8000/generate", json=test_payload, timeout=90)
print(response.status_code)
print(response.json())

## 7. Creating The Public Tunnel

Then copying the printed `COLAB_LLM_URL` into your local `.env` file.


In [ ]:
import re
import subprocess
import time

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(120):
    line = proc.stdout.readline()
    print(line, end="")
    match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.2)

print("\nCOLAB_LLM_URL=", public_url)
print("COLAB_LLM_TIMEOUT=90")